# Lab 1 - NumPy warm-up and an honest baseline

**Session 1, Foundations of Machine Learning (Demo).** Work through the cells, then do the
three exercises at the end. Everything you write here you will reuse in later labs.

Goals: vectorised arithmetic, broadcasting, a train/test split by index, and a metric
function of your own.

## 1. The data

In [ ]:
# The twelve flats from the lectures. The same rows are released as
# data/housing-mini.csv in the cohort materials repo.
import numpy as np

area = np.array([32, 45, 52, 60, 68, 75, 80, 95, 38, 55, 110, 48], float)
dist = np.array([0.3, 0.9, 0.4, 1.6, 0.7, 2.1, 1.1, 0.5, 1.8, 0.6, 1.4, 2.6])
rent = np.array([540, 510, 640, 545, 720, 620, 770, 860, 420, 640, 930, 400], float)
print(area.shape, dist.shape, rent.shape)

## 2. Vectorised arithmetic

NumPy applies operations elementwise, so a loop is almost never needed. `rent / area` is
one operation on twelve pairs, not twelve operations.

In [ ]:
rent_per_sqm = rent / area
print(np.round(rent_per_sqm, 2))
print(f"mean {rent_per_sqm.mean():.2f}  sd {rent_per_sqm.std(ddof=1):.2f}  "
      f"min {rent_per_sqm.min():.2f}  max {rent_per_sqm.max():.2f}")

## 3. Broadcasting

A `(12, 1)` column and a `(3,)` row combine into a `(12, 3)` grid. Broadcasting is what
lets you evaluate many candidate models at once - we use it for a learning-rate grid in
lab 3.

In [ ]:
candidate_slopes = np.array([5.0, 6.0, 7.0])
predictions = area[:, None] * candidate_slopes      # (12, 1) * (3,) -> (12, 3)
print(predictions.shape)
sse = ((rent[:, None] - predictions) ** 2).sum(axis=0)
for slope, s in zip(candidate_slopes, sse):
    print(f"slope {slope:.1f}  SSE {s:12.1f}")

## 4. A split, decided before looking at the target

Eight rows to train on, four held back. `np.random.default_rng(seed)` is the modern way to
get a reproducible generator - never the global `np.random.seed`.

In [ ]:
rng = np.random.default_rng(2026)
idx = rng.permutation(len(rent))
train_idx, test_idx = idx[:8], idx[8:]
print("train rows:", train_idx, "\ntest  rows:", test_idx)

## 5. A metric of your own

In [ ]:
def mae(actual, predicted):
    """Mean absolute error - in euros, so a reader can interpret it."""
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(predicted))))


baseline = rent[train_idx].mean()
print(f"baseline (train mean) = {baseline:.1f} EUR")
print(f"  train MAE = {mae(rent[train_idx], baseline):6.1f} EUR")
print(f"  test  MAE = {mae(rent[test_idx], baseline):6.1f} EUR")

## 6. One feature, using `lstsq`

`np.linalg.lstsq` solves least squares properly (via SVD). Build a design matrix with an
intercept column, fit on the training rows only, then score both splits.

In [ ]:
def design(x):
    return np.column_stack([np.ones_like(x), x])


coef, *_ = np.linalg.lstsq(design(area[train_idx]), rent[train_idx], rcond=None)
print(f"rent_hat = {coef[0]:.1f} + {coef[1]:.2f} * area_sqm")
print(f"  train MAE = {mae(rent[train_idx], design(area[train_idx]) @ coef):6.1f} EUR")
print(f"  test  MAE = {mae(rent[test_idx], design(area[test_idx]) @ coef):6.1f} EUR")

## Exercises

1. **Two features.** Add `dist` to the design matrix and refit. Does the test MAE improve?
   By enough to justify the extra column, given that there are only four test rows?
2. **RMSE.** Write `rmse(actual, predicted)` and compare it with MAE on the same fits.
   Which is larger here, and why must that always be so?
3. **Split sensitivity.** Re-run cells 4-6 with seeds 1, 2 and 3. How much does the test MAE
   move? Write one sentence on what that implies about reporting a single number from a
   four-row test set - this is the point session 10 returns to.

Push nothing: labs are not graded. But keep your `mae` and `rmse` - lab 3 imports the same
idea.